In [ ]:
import pandas as pd
import numpy as np
# Load biomarker table
# biomarker_df = pd.read_csv(r"output/extracted_biomarkers.csv")
biomarker_df = pd.read_csv(r'output_ablation/no_diffusion/EXTRACTED_BIOMARKERS_NO_DIFF.csv')
all_gene_name=biomarker_df["Gene_Symbol"]
unique_gene_name=all_gene_name
# Load OncoKB™ Cancer Gene List
oncokb_df=pd.read_table(r"C:\Users\91900\Downloads\cancerGeneList.tsv")
oncokb_genes=oncokb_df['Hugo Symbol'].unique()
literature_validated_genes=[gene for gene in unique_gene_name if gene in oncokb_genes]
# print(filterd_oncokb_information)
oncokb_df_final=biomarker_df[biomarker_df["Gene_Symbol"].isin(literature_validated_genes)]
# oncokb_df_final.to_csv(r"D:\Mcode final\output\oncoKB_biomarkers.csv")
# oncokb_df_final.to_csv(r"output\oncoKB_biomarkers.csv")
# print(oncokb_df_final)


Total number of top 199 genes matches in OncoKB database are 25: ['ALB', 'AURKB', 'BRCA1', 'BRCA2', 'BUB1B', 'CACNA1D', 'CHEK1', 'DDR2', 'DKK1', 'DUSP4', 'FANCA', 'FANCD2', 'H1-2', 'H2AC16', 'H2BC17', 'H3-5', 'H3C12', 'H3C2', 'IGF1', 'MAD2L2', 'NTHL1', 'NUF2', 'PC', 'RECQL4', 'SEPTIN6']


# 1 hop expansion

In [5]:
import pandas as pd

# 1. Load your datasets
df_total = pd.read_csv('output/extracted_biomarkers.csv')
df_onco = pd.read_csv('output/oncoKB_biomarkers.csv')

# Standard cleaning to remove Ensembl version decimals (e.g., .13)
def clean_id(x): return str(x).split('.')[0]
df_total['Clean_ID'] = df_total['Gene'].apply(clean_id)
df_onco['Clean_ID'] = df_onco['Gene'].apply(clean_id)

# Define our sets
onco_seeds = set(df_onco['Clean_ID'])
candidate_pool = set(df_total[~df_total['Clean_ID'].isin(onco_seeds)]['Clean_ID'])

# 2. Load the PPI Network
ppi = pd.read_csv(r"D:\Mcode final__LUAD\data\ppi_network.csv") 

# Filter for High Confidence
high_conf = ppi[ppi['Confidence_Score'] >= 0.7].copy()

# Apply cleaning to PPI protein columns to remove version decimals
high_conf['Protein1'] = high_conf['Protein1'].apply(clean_id)
high_conf['Protein2'] = high_conf['Protein2'].apply(clean_id)

# 3. 1-Hop Expansion (Neighborhood Validation)
# Keep a candidate if it interacts with at least one OncoKB seed
match_a = high_conf[high_conf['Protein1'].isin(onco_seeds) & high_conf['Protein2'].isin(candidate_pool)]
match_b = high_conf[high_conf['Protein2'].isin(onco_seeds) & high_conf['Protein1'].isin(candidate_pool)]

# Get unique Ensembl IDs of validated novel biomarkers
validated_candidate_ids = set(match_a['Protein2']).union(set(match_b['Protein1']))

# 4. Compile the Final Panel
final_panel = df_total[df_total['Clean_ID'].isin(onco_seeds) | df_total['Clean_ID'].isin(validated_candidate_ids)]

# Save the final result
# final_panel.to_csv('final_validated_biomarker_panel.csv', index=False)

print(f"Analysis Complete:")
print(f"- OncoKB Seeds: {len(onco_seeds)}")
print(f"- Novel Biomarkers Validated (1-hop neighbors): {len(validated_candidate_ids)}")
print(f"- Total Validated Panel: {len(final_panel)}")

Analysis Complete:
- OncoKB Seeds: 20
- Novel Biomarkers Validated (1-hop neighbors): 43
- Total Validated Panel: 63


In [6]:
final_panel.to_csv('output/final_validated_biomarker_panel.csv', index=False)
